# Week 80, Day 5: Project: deploy

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w080d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('miniflask.py', 'w', encoding='utf-8') as f:
    f.write('"""A small subset of Flask that runs in the page (Week 77).\n\nThe names and behaviour follow Flask\'s own, so code written against this module also runs with\nthe real Flask (pip install flask): Flask, request, g, jsonify, abort, redirect, make_response,\nurl_for, Blueprint, and app.test_client(). It\'s a teaching copy, not the real thing: no templates,\nsessions, streaming or debugger.\n"""\nimport io\nimport json\nimport re\nfrom contextvars import ContextVar\nfrom http import HTTPStatus\nfrom urllib.parse import parse_qs, urlencode, urlsplit\n\n_current = ContextVar("request")\n\n\nclass HTTPException(Exception):\n    def __init__(self, code, description=None):\n        self.code = code\n        self.name = HTTPStatus(code).phrase\n        self.description = description or self.name\n        super().__init__(f"{code} {self.name}: {self.description}")\n\n\ndef abort(code, description=None):\n    raise HTTPException(code, description)\n\n\nclass Headers:\n    """Case-insensitive headers."""\n    def __init__(self, items=()):\n        self._items = [(k, str(v)) for k, v in (items.items() if isinstance(items, dict) else items)]\n\n    def get(self, name, default=None):\n        for k, v in self._items:\n            if k.lower() == name.lower():\n                return v\n        return default\n\n    def __getitem__(self, name):\n        value = self.get(name)\n        if value is None:\n            raise KeyError(name)\n        return value\n\n    def __contains__(self, name):\n        return self.get(name) is not None\n\n    def __setitem__(self, name, value):\n        self._items = [(k, v) for k, v in self._items if k.lower() != name.lower()] + [(name, str(value))]\n\n    def items(self):\n        return list(self._items)\n\n\nclass Args(dict):\n    """Query or form values: .get(name) gives the first value; .getlist(name) all of them."""\n    def __init__(self, pairs):\n        super().__init__()\n        self._lists = {}\n        for k, v in pairs:\n            self._lists.setdefault(k, []).append(v)\n            dict.setdefault(self, k, v)\n\n    def get(self, key, default=None, type=None):\n        if key not in self:\n            return default\n        value = self[key]\n        if type is not None:\n            try:\n                return type(value)\n            except (TypeError, ValueError):\n                return default\n        return value\n\n    def getlist(self, key):\n        return list(self._lists.get(key, []))\n\n\nclass Request:\n    def __init__(self, environ):\n        self.environ = environ\n        self.method = environ["REQUEST_METHOD"].upper()\n        self.path = environ.get("PATH_INFO", "/") or "/"\n        self.args = Args(_pairs(environ.get("QUERY_STRING", "")))\n        items = [(k[5:].replace("_", "-").title(), v) for k, v in environ.items() if k.startswith("HTTP_")]\n        if environ.get("CONTENT_TYPE"):\n            items.append(("Content-Type", environ["CONTENT_TYPE"]))\n        self.headers = Headers(items)\n        length = int(environ.get("CONTENT_LENGTH") or 0)\n        self.data = environ["wsgi.input"].read(length) if length else b""\n        self.content_type = environ.get("CONTENT_TYPE", "")\n        self.form = Args(_pairs(self.data.decode("utf-8"))) if self.content_type.startswith("application/x-www-form-urlencoded") else Args([])\n        self.view_args = {}\n        cookie = environ.get("HTTP_COOKIE", "")\n        self.cookies = dict(p.strip().split("=", 1) for p in cookie.split(";") if "=" in p)\n\n    @property\n    def is_json(self):\n        return self.content_type.split(";")[0].strip() == "application/json"\n\n    def get_json(self, silent=False, force=False):\n        if not (self.is_json or force):\n            if silent:\n                return None\n            abort(415, "expected a JSON body (Content-Type: application/json)")\n        try:\n            return json.loads(self.data or b"null")\n        except ValueError:\n            if silent:\n                return None\n            abort(400, "the body isn\'t valid JSON")\n\n    @property\n    def json(self):\n        return self.get_json()\n\n\ndef _pairs(text):\n    return [(k, v) for k, vs in parse_qs(text, keep_blank_values=True).items() for v in vs]\n\n\nclass _Proxy:\n    def __init__(self, getter):\n        object.__setattr__(self, "_getter", getter)\n\n    def __getattr__(self, name):\n        return getattr(self._getter(), name)\n\n    def __setattr__(self, name, value):\n        setattr(self._getter(), name, value)\n\n\ndef _request():\n    try:\n        return _current.get()[0]\n    except LookupError:\n        raise RuntimeError("working outside of request context") from None\n\n\ndef _g():\n    try:\n        return _current.get()[1]\n    except LookupError:\n        raise RuntimeError("working outside of request context") from None\n\n\nclass _G:\n    def get(self, name, default=None):\n        return self.__dict__.get(name, default)\n\n\nrequest = _Proxy(_request)\ng = _Proxy(_g)\n\n\nclass Response:\n    def __init__(self, response=b"", status=200, headers=None, mimetype=None, content_type=None):\n        if isinstance(response, str):\n            response = response.encode("utf-8")\n        self.data = response\n        self.status_code = status\n        self.headers = Headers(headers or {})\n        if content_type or mimetype or "Content-Type" not in self.headers:\n            ctype = content_type or (f"{mimetype}; charset=utf-8" if mimetype and mimetype.startswith("text/") else mimetype) or "text/html; charset=utf-8"\n            self.headers["Content-Type"] = ctype\n\n    @property\n    def status(self):\n        return f"{self.status_code} {HTTPStatus(self.status_code).phrase}"\n\n    @property\n    def text(self):\n        return self.data.decode("utf-8")\n\n    def get_json(self):\n        return json.loads(self.data)\n\n    @property\n    def json(self):\n        return self.get_json()\n\n    def set_cookie(self, key, value="", max_age=None, path="/", secure=False, httponly=False, samesite=None):\n        parts = [f"{key}={value}"]\n        if max_age is not None:\n            parts.append(f"Max-Age={max_age}")\n        parts.append(f"Path={path}")\n        if secure:\n            parts.append("Secure")\n        if httponly:\n            parts.append("HttpOnly")\n        if samesite:\n            parts.append(f"SameSite={samesite}")\n        self.headers._items.append(("Set-Cookie", "; ".join(parts)))\n\n\ndef jsonify(*args, **kwargs):\n    data = args[0] if len(args) == 1 else (list(args) if args else kwargs)\n    return Response(json.dumps(data), headers={"Content-Type": "application/json"})\n\n\ndef redirect(location, code=302):\n    return Response(b"", code, {"Location": location})\n\n\ndef make_response(*args):\n    rv = args[0] if len(args) == 1 else args\n    return _App._to_response(rv)\n\n\ndef url_for(endpoint, **values):\n    return _current.get()[2].url_for(endpoint, **values)\n\n\n_CONVERTERS = {"int": (r"\\d+", int), "float": (r"\\d+(?:\\.\\d+)?", float), "path": (r".+", str), "string": (r"[^/]+", str)}\n\n\nclass _Rule:\n    def __init__(self, rule, endpoint, view, methods):\n        self.rule, self.endpoint, self.view = rule, endpoint, view\n        self.methods = {m.upper() for m in methods} | ({"HEAD"} if "GET" in {m.upper() for m in methods} else set())\n        regex, self.converters = "", {}\n        for piece in re.split(r"(<[^>]+>)", rule):\n            m = re.fullmatch(r"<(?:(\\w+):)?(\\w+)>", piece)\n            if m:\n                kind, name = m.group(1) or "string", m.group(2)\n                pattern, convert = _CONVERTERS[kind]\n                regex += f"(?P<{name}>{pattern})"\n                self.converters[name] = convert\n            else:\n                regex += re.escape(piece)\n        self.regex = re.compile(regex)\n\n    def match(self, path):\n        m = self.regex.fullmatch(path)\n        if not m:\n            return None\n        return {k: self.converters[k](v) for k, v in m.groupdict().items()}\n\n    def build(self, values):\n        def fill(m):\n            return str(values.pop(m.group(2)))\n        path = re.sub(r"<(?:(\\w+):)?(\\w+)>", fill, self.rule)\n        return path + (f"?{urlencode(values)}" if values else "")\n\n\nclass _Setup:\n    def route(self, rule, methods=("GET",), endpoint=None):\n        def decorate(view):\n            self._add(rule, endpoint or view.__name__, view, methods)\n            return view\n        return decorate\n\n    def get(self, rule, **kw):\n        return self.route(rule, methods=["GET"], **kw)\n\n    def post(self, rule, **kw):\n        return self.route(rule, methods=["POST"], **kw)\n\n    def put(self, rule, **kw):\n        return self.route(rule, methods=["PUT"], **kw)\n\n    def patch(self, rule, **kw):\n        return self.route(rule, methods=["PATCH"], **kw)\n\n    def delete(self, rule, **kw):\n        return self.route(rule, methods=["DELETE"], **kw)\n\n\nclass Blueprint(_Setup):\n    def __init__(self, name, import_name, url_prefix=""):\n        self.name, self.url_prefix = name, url_prefix\n        self.deferred = []\n\n    def _add(self, rule, endpoint, view, methods):\n        self.deferred.append((rule, endpoint, view, methods))\n\n\nclass _App(_Setup):\n    def __init__(self, import_name):\n        self.import_name = import_name\n        self.config = {"TESTING": False}\n        self.rules = []\n        self.before_funcs, self.after_funcs, self.error_handlers = [], [], {}\n\n    def _add(self, rule, endpoint, view, methods):\n        self.rules.append(_Rule(rule, endpoint, view, methods))\n\n    def register_blueprint(self, bp, url_prefix=None):\n        prefix = url_prefix if url_prefix is not None else bp.url_prefix\n        for rule, endpoint, view, methods in bp.deferred:\n            self._add(prefix + rule, f"{bp.name}.{endpoint}", view, methods)\n\n    def before_request(self, f):\n        self.before_funcs.append(f)\n        return f\n\n    def after_request(self, f):\n        self.after_funcs.append(f)\n        return f\n\n    def errorhandler(self, code):\n        def decorate(f):\n            self.error_handlers[code] = f\n            return f\n        return decorate\n\n    def url_for(self, endpoint, **values):\n        for r in self.rules:\n            if r.endpoint == endpoint:\n                return r.build(dict(values))\n        raise LookupError(f"no endpoint {endpoint!r}")\n\n    @staticmethod\n    def _to_response(rv):\n        status, headers = None, None\n        if isinstance(rv, tuple):\n            if len(rv) == 3:\n                rv, status, headers = rv\n            elif isinstance(rv[1], (dict, list)):\n                rv, headers = rv\n            else:\n                rv, status = rv\n        if isinstance(rv, Response):\n            response = rv\n        elif isinstance(rv, (dict, list)):\n            response = jsonify(rv)\n        elif isinstance(rv, (str, bytes)):\n            response = Response(rv)\n        elif rv is None:\n            raise TypeError("the view function did not return a valid response")\n        else:\n            raise TypeError(f"can\'t turn {type(rv).__name__} into a response")\n        if status is not None:\n            response.status_code = status\n        for k, v in (headers.items() if isinstance(headers, dict) else headers or []):\n            response.headers[k] = v\n        return response\n\n    def _error(self, e):\n        handler = self.error_handlers.get(e.code)\n        if handler:\n            return self._to_response(handler(e))\n        body = f"<h1>{e.code} {e.name}</h1><p>{e.description}</p>"\n        return Response(body, e.code)\n\n    def _dispatch(self, req):\n        path = req.path\n        allowed = set()\n        for r in self.rules:\n            values = r.match(path)\n            if values is None:\n                continue\n            if req.method in r.methods:\n                req.view_args = values\n                return r.view(**values)\n            allowed |= r.methods\n        if allowed:\n            raise _MethodNotAllowed(sorted(allowed | {"OPTIONS"}))\n        abort(404, "The requested URL was not found on the server.")\n\n    def full_dispatch(self, environ):\n        req = Request(environ)\n        token = _current.set((req, _G(), self))\n        try:\n            try:\n                rv = None\n                for f in self.before_funcs:\n                    rv = f()\n                    if rv is not None:\n                        break\n                if rv is None:\n                    rv = self._dispatch(req)\n                response = self._to_response(rv)\n            except _MethodNotAllowed as e:\n                response = self._error(HTTPException(405))\n                response.headers["Allow"] = ", ".join(e.allowed)\n            except HTTPException as e:\n                response = self._error(e)\n            except Exception as e:\n                if self.config.get("TESTING") and 500 not in self.error_handlers:\n                    raise\n                response = self._error(HTTPException(500)) if 500 not in self.error_handlers else self._to_response(self.error_handlers[500](e))\n                response.status_code = 500\n            for f in self.after_funcs:\n                response = f(response) or response\n            if req.method == "HEAD":\n                response.headers["Content-Length"] = len(response.data)\n                response.data = b""\n            return response\n        finally:\n            _current.reset(token)\n\n    def __call__(self, environ, start_response):\n        response = self.full_dispatch(environ)\n        if "Content-Length" not in response.headers:\n            response.headers["Content-Length"] = len(response.data)\n        start_response(response.status, response.headers.items())\n        return [response.data]\n\n    def test_client(self):\n        return _TestClient(self)\n\n\nclass _MethodNotAllowed(Exception):\n    def __init__(self, allowed):\n        self.allowed = allowed\n\n\nclass Flask(_App):\n    pass\n\n\nclass _TestClient:\n    def __init__(self, app):\n        self.app = app\n        self.cookies = {}\n\n    def open(self, path, method="GET", json=None, data=None, headers=None, query_string=None):\n        url = urlsplit(path)\n        query = url.query\n        if query_string:\n            query = (query + "&" if query else "") + urlencode(query_string, doseq=True)\n        body, ctype = b"", None\n        if json is not None:\n            import json as _json\n            body, ctype = _json.dumps(json).encode("utf-8"), "application/json"\n        elif isinstance(data, dict):\n            body, ctype = urlencode(data, doseq=True).encode("utf-8"), "application/x-www-form-urlencoded"\n        elif data is not None:\n            body = data.encode("utf-8") if isinstance(data, str) else data\n        environ = {"REQUEST_METHOD": method.upper(), "PATH_INFO": url.path or "/", "QUERY_STRING": query,\n                   "SERVER_NAME": "localhost", "SERVER_PORT": "80", "wsgi.url_scheme": "http",\n                   "wsgi.input": io.BytesIO(body), "CONTENT_LENGTH": str(len(body))}\n        headers = dict(headers or {})\n        if ctype and not any(k.lower() == "content-type" for k in headers):\n            environ["CONTENT_TYPE"] = ctype\n        for k, v in headers.items():\n            if k.lower() == "content-type":\n                environ["CONTENT_TYPE"] = v\n            else:\n                environ["HTTP_" + k.upper().replace("-", "_")] = v\n        if self.cookies:\n            environ["HTTP_COOKIE"] = "; ".join(f"{k}={v}" for k, v in self.cookies.items())\n        response = self.app.full_dispatch(environ)\n        for k, v in response.headers.items():\n            if k.lower() == "set-cookie":\n                name, _, rest = v.partition("=")\n                self.cookies[name] = rest.split(";")[0]\n        return response\n\n    def get(self, path, **kw):\n        return self.open(path, "GET", **kw)\n\n    def post(self, path, **kw):\n        return self.open(path, "POST", **kw)\n\n    def put(self, path, **kw):\n        return self.open(path, "PUT", **kw)\n\n    def patch(self, path, **kw):\n        return self.open(path, "PATCH", **kw)\n\n    def delete(self, path, **kw):\n        return self.open(path, "DELETE", **kw)\n')
with open('helpdesk_domain.py', 'w', encoding='utf-8') as f:
    f.write('"""The helpdesk\'s rules, with no web or database code (Week 80, Day 1).\n\nEverything here is plain data and pure functions, so it can be tested on its own and used by the\ndatabase layer, the web layer and the tests alike.\n"""\nimport re\nfrom http import HTTPStatus\n\nROLES = ("customer", "agent", "admin")\nPRIORITIES = ("low", "normal", "high", "urgent")\nSTATUSES = ("open", "in_progress", "waiting", "resolved", "closed")\nSTAFF = {"agent", "admin"}\n\n# status -> {new status: roles that may make the change}\nTRANSITIONS = {\n    "open": {"in_progress": {"agent", "admin"}, "closed": {"customer", "admin"}},\n    "in_progress": {"waiting": {"agent", "admin"}, "resolved": {"agent", "admin"}},\n    "waiting": {"in_progress": {"customer", "agent", "admin"}},\n    "resolved": {"closed": {"customer", "admin"}, "in_progress": {"customer", "agent", "admin"}},\n    "closed": {},\n}\n\n# action -> (roles always allowed, roles allowed only on their own tickets)\nPERMISSIONS = {\n    "create_ticket": ({"customer"}, set()),\n    "view_ticket": ({"agent", "admin"}, {"customer"}),\n    "list_all_tickets": ({"agent", "admin"}, set()),\n    "edit_ticket": ({"admin"}, {"customer"}),\n    "comment": ({"agent", "admin"}, {"customer"}),\n    "internal_note": ({"agent", "admin"}, set()),\n    "change_priority": ({"agent", "admin"}, set()),\n    "assign": ({"agent", "admin"}, set()),\n    "delete_ticket": ({"admin"}, set()),\n    "view_stats": ({"agent", "admin"}, set()),\n}\n\n\nclass ValidationError(Exception):\n    """Raised with a dict of field -> message."""\n\n    def __init__(self, errors):\n        super().__init__("; ".join(f"{k}: {v}" for k, v in sorted(errors.items())))\n        self.errors = errors\n\n\ndef validate_ticket(data, partial=False):\n    """Check a ticket body from a client. partial=True for PATCH: only the fields present are checked."""\n    if not isinstance(data, dict):\n        raise ValidationError({"_": "expected a JSON object"})\n    errors, clean = {}, {}\n    unknown = sorted(set(data) - {"title", "body", "priority"})\n    if unknown:\n        errors["_"] = f"unexpected field(s): {\', \'.join(unknown)}"\n    for field in ("title", "body"):\n        if field not in data:\n            if not partial:\n                errors[field] = "required"\n            continue\n        value = data[field]\n        limit = (3, 120) if field == "title" else (1, 5000)\n        if not isinstance(value, str) or not limit[0] <= len(value.strip()) <= limit[1]:\n            errors[field] = f"{limit[0]} to {limit[1]} characters"\n        else:\n            clean[field] = value.strip()\n    if "priority" in data:\n        if data["priority"] not in PRIORITIES:\n            errors["priority"] = f"one of {\', \'.join(PRIORITIES)}"\n        else:\n            clean["priority"] = data["priority"]\n    elif not partial:\n        clean["priority"] = "normal"\n    if partial and not data:\n        errors["_"] = "nothing to change"\n    if errors:\n        raise ValidationError(errors)\n    return clean\n\n\ndef can_transition(current, new, role):\n    """True if someone with this role may move a ticket from `current` to `new`."""\n    return role in TRANSITIONS.get(current, {}).get(new, set())\n\n\ndef allowed(role, action, is_owner=False):\n    """True if the role may do the action (is_owner: the ticket is the user\'s own)."""\n    always, own = PERMISSIONS[action]\n    return role in always or (is_owner and role in own)\n\n\ndef problem(status, detail, title=None, **extra):\n    """An RFC 9457 problem-details response: (body, status, headers)."""\n    body = {"type": "about:blank", "title": title or HTTPStatus(status).phrase, "status": status, "detail": detail}\n    body.update(extra)\n    return body, status, {"Content-Type": "application/problem+json"}\n\n\nROUTES = [\n    ("POST", "/users", "sign up as a customer", 201),\n    ("POST", "/tokens", "log in and get a token", 201),\n    ("GET", "/me", "who am I", 200),\n    ("GET", "/tickets", "list tickets (customers see their own)", 200),\n    ("POST", "/tickets", "open a ticket", 201),\n    ("GET", "/tickets/{ticket_id}", "read a ticket", 200),\n    ("PATCH", "/tickets/{ticket_id}", "change a ticket\'s title, body or priority", 200),\n    ("DELETE", "/tickets/{ticket_id}", "delete a ticket (admins)", 204),\n    ("POST", "/tickets/{ticket_id}/transitions", "change a ticket\'s status", 200),\n    ("PUT", "/tickets/{ticket_id}/assignee", "assign a ticket to an agent", 200),\n    ("GET", "/tickets/{ticket_id}/comments", "list a ticket\'s comments", 200),\n    ("POST", "/tickets/{ticket_id}/comments", "add a comment or internal note", 201),\n    ("GET", "/stats", "ticket counts (staff)", 200),\n    ("GET", "/healthz", "is the process alive", 200),\n    ("GET", "/readyz", "can it serve requests", 200),\n]\n\n\ndef docs_table(routes):\n    """The routes as a Markdown table, for the README."""\n    lines = ["| Method | Path | What it does | Success |", "|---|---|---|---|"]\n    lines += [f"| {m} | `{p}` | {what} | {code} {HTTPStatus(code).phrase} |" for m, p, what, code in routes]\n    return "\\n".join(lines)\n')
with open('helpdesk_db.py', 'w', encoding='utf-8') as f:
    f.write('"""The helpdesk\'s database: migrations and the queries the app needs (Week 80, Day 2).\n\nEvery query uses parameters; every change that touches more than one row runs in a transaction.\n"""\nimport sqlite3\n\nMIGRATIONS = [\n    (1, "users", """\n        CREATE TABLE users (\n            id INTEGER PRIMARY KEY,\n            email TEXT NOT NULL UNIQUE,\n            password TEXT NOT NULL,\n            role TEXT NOT NULL CHECK (role IN (\'customer\', \'agent\', \'admin\')),\n            created_at TEXT NOT NULL\n        )"""),\n    (2, "tickets", """\n        CREATE TABLE tickets (\n            id INTEGER PRIMARY KEY,\n            customer_id INTEGER NOT NULL REFERENCES users (id),\n            assignee_id INTEGER REFERENCES users (id),\n            title TEXT NOT NULL,\n            body TEXT NOT NULL,\n            priority TEXT NOT NULL DEFAULT \'normal\' CHECK (priority IN (\'low\', \'normal\', \'high\', \'urgent\')),\n            status TEXT NOT NULL DEFAULT \'open\'\n                CHECK (status IN (\'open\', \'in_progress\', \'waiting\', \'resolved\', \'closed\')),\n            created_at TEXT NOT NULL,\n            updated_at TEXT NOT NULL\n        )"""),\n    (3, "comments and events", """\n        CREATE TABLE comments (\n            id INTEGER PRIMARY KEY,\n            ticket_id INTEGER NOT NULL REFERENCES tickets (id) ON DELETE CASCADE,\n            author_id INTEGER NOT NULL REFERENCES users (id),\n            body TEXT NOT NULL,\n            internal INTEGER NOT NULL DEFAULT 0,\n            created_at TEXT NOT NULL\n        );\n        CREATE TABLE events (\n            id INTEGER PRIMARY KEY,\n            ticket_id INTEGER NOT NULL REFERENCES tickets (id) ON DELETE CASCADE,\n            actor_id INTEGER NOT NULL REFERENCES users (id),\n            kind TEXT NOT NULL,\n            detail TEXT NOT NULL,\n            at TEXT NOT NULL\n        )"""),\n    (4, "indexes", """\n        CREATE INDEX idx_tickets_status ON tickets (status, id);\n        CREATE INDEX idx_tickets_customer ON tickets (customer_id, id);\n        CREATE INDEX idx_comments_ticket ON comments (ticket_id, id)"""),\n]\n\n\nclass ConflictError(Exception):\n    """The row changed since it was read."""\n\n\ndef connect(path=":memory:"):\n    conn = sqlite3.connect(path)\n    conn.row_factory = sqlite3.Row\n    conn.execute("PRAGMA foreign_keys = ON")\n    return conn\n\n\ndef migrate(conn, migrations=MIGRATIONS):\n    """Apply the migrations the database hasn\'t had yet, each in its own transaction. Returns their versions."""\n    conn.execute("CREATE TABLE IF NOT EXISTS schema_migrations (version INTEGER PRIMARY KEY, name TEXT NOT NULL)")\n    conn.commit()\n    versions = [v for v, _, _ in migrations]\n    if versions != list(range(1, len(versions) + 1)):\n        raise ValueError("migrations must be numbered 1, 2, 3... with no gaps")\n    current = conn.execute("SELECT COALESCE(MAX(version), 0) FROM schema_migrations").fetchone()[0]\n    if current > len(versions):\n        raise RuntimeError(f"the database is at version {current}, newer than this code")\n    applied = []\n    for version, name, sql in migrations[current:]:\n        try:\n            conn.execute("BEGIN")\n            for statement in sql.split(";"):\n                if statement.strip():\n                    conn.execute(statement)\n            conn.execute("INSERT INTO schema_migrations (version, name) VALUES (?, ?)", (version, name))\n            conn.commit()\n        except Exception:\n            conn.rollback()\n            raise\n        applied.append(version)\n    return applied\n\n\ndef _dict(row):\n    return dict(row) if row is not None else None\n\n\ndef _event(conn, ticket_id, actor_id, kind, detail, now):\n    conn.execute("INSERT INTO events (ticket_id, actor_id, kind, detail, at) VALUES (?, ?, ?, ?, ?)",\n                 (ticket_id, actor_id, kind, detail, now))\n\n\ndef create_user(conn, email, password_hash, role, now):\n    try:\n        with conn:\n            cur = conn.execute("INSERT INTO users (email, password, role, created_at) VALUES (?, ?, ?, ?)",\n                               (email, password_hash, role, now))\n    except sqlite3.IntegrityError as e:\n        if "UNIQUE" not in str(e):\n            raise\n        raise ValueError("email already registered") from None\n    return cur.lastrowid\n\n\ndef get_user(conn, user_id):\n    return _dict(conn.execute("SELECT id, email, role, created_at FROM users WHERE id = ?", (user_id,)).fetchone())\n\n\ndef get_user_by_email(conn, email):\n    return _dict(conn.execute("SELECT id, email, password, role FROM users WHERE email = ?", (email,)).fetchone())\n\n\ndef create_ticket(conn, customer_id, clean, now):\n    with conn:\n        cur = conn.execute(\n            "INSERT INTO tickets (customer_id, title, body, priority, created_at, updated_at) VALUES (?, ?, ?, ?, ?, ?)",\n            (customer_id, clean["title"], clean["body"], clean.get("priority", "normal"), now, now))\n        _event(conn, cur.lastrowid, customer_id, "created", clean["title"], now)\n    return get_ticket(conn, cur.lastrowid)\n\n\ndef get_ticket(conn, ticket_id):\n    return _dict(conn.execute("SELECT * FROM tickets WHERE id = ?", (ticket_id,)).fetchone())\n\n\ndef _like(text):\n    return "%" + text.replace("\\\\", "\\\\\\\\").replace("%", "\\\\%").replace("_", "\\\\_") + "%"\n\n\ndef list_tickets(conn, customer_id=None, status=None, priority=None, assignee_id=None, q=None, after_id=0, limit=20):\n    """Tickets matching the filters, by id, after a cursor. Returns (items, next_after or None)."""\n    where, values = ["id > ?"], [after_id]\n    for column, value in (("customer_id", customer_id), ("status", status), ("priority", priority), ("assignee_id", assignee_id)):\n        if value is not None:\n            where.append(f"{column} = ?")\n            values.append(value)\n    if q:\n        where.append("title LIKE ? ESCAPE \'\\\\\'")\n        values.append(_like(q))\n    rows = conn.execute(f"SELECT * FROM tickets WHERE {\' AND \'.join(where)} ORDER BY id LIMIT ?", values + [limit + 1]).fetchall()\n    items = [dict(r) for r in rows[:limit]]\n    return items, (items[-1]["id"] if len(rows) > limit else None)\n\n\ndef update_ticket(conn, ticket_id, changes, actor_id, now):\n    if not changes:\n        return get_ticket(conn, ticket_id)\n    columns = sorted(changes)\n    with conn:\n        conn.execute(f"UPDATE tickets SET {\', \'.join(f\'{c} = ?\' for c in columns)}, updated_at = ? WHERE id = ?",\n                     [changes[c] for c in columns] + [now, ticket_id])\n        _event(conn, ticket_id, actor_id, "edited", ",".join(columns), now)\n    return get_ticket(conn, ticket_id)\n\n\ndef change_status(conn, ticket_id, expected, new, actor_id, now):\n    """Move a ticket from `expected` to `new`, unless someone changed its status first."""\n    with conn:\n        cur = conn.execute("UPDATE tickets SET status = ?, updated_at = ? WHERE id = ? AND status = ?",\n                           (new, now, ticket_id, expected))\n        if cur.rowcount != 1:\n            raise ConflictError(f"ticket {ticket_id} is no longer {expected}")\n        _event(conn, ticket_id, actor_id, "status", f"{expected}->{new}", now)\n    return get_ticket(conn, ticket_id)\n\n\ndef assign(conn, ticket_id, agent_id, actor_id, now):\n    agent = get_user(conn, agent_id)\n    if agent is None or agent["role"] not in ("agent", "admin"):\n        raise ValueError(f"user {agent_id} is not an agent")\n    with conn:\n        conn.execute("UPDATE tickets SET assignee_id = ?, updated_at = ? WHERE id = ?", (agent_id, now, ticket_id))\n        _event(conn, ticket_id, actor_id, "assigned", str(agent_id), now)\n    return get_ticket(conn, ticket_id)\n\n\ndef add_comment(conn, ticket_id, author_id, body, internal, now):\n    with conn:\n        cur = conn.execute("INSERT INTO comments (ticket_id, author_id, body, internal, created_at) VALUES (?, ?, ?, ?, ?)",\n                           (ticket_id, author_id, body, int(internal), now))\n        conn.execute("UPDATE tickets SET updated_at = ? WHERE id = ?", (now, ticket_id))\n    return _dict(conn.execute("SELECT * FROM comments WHERE id = ?", (cur.lastrowid,)).fetchone())\n\n\ndef list_comments(conn, ticket_id, include_internal):\n    sql = "SELECT * FROM comments WHERE ticket_id = ?" + ("" if include_internal else " AND internal = 0") + " ORDER BY id"\n    return [dict(r) for r in conn.execute(sql, (ticket_id,)).fetchall()]\n\n\ndef delete_ticket(conn, ticket_id):\n    with conn:\n        return conn.execute("DELETE FROM tickets WHERE id = ?", (ticket_id,)).rowcount == 1\n\n\ndef events(conn, ticket_id):\n    return [dict(r) for r in conn.execute("SELECT kind, detail, actor_id, at FROM events WHERE ticket_id = ? ORDER BY id", (ticket_id,))]\n\n\ndef stats(conn):\n    by_status = {s: 0 for s in ("open", "in_progress", "waiting", "resolved", "closed")}\n    by_status.update(dict(conn.execute("SELECT status, COUNT(*) FROM tickets GROUP BY status").fetchall()))\n    by_priority = {p: 0 for p in ("low", "normal", "high", "urgent")}\n    by_priority.update(dict(conn.execute("SELECT priority, COUNT(*) FROM tickets WHERE status != \'closed\' GROUP BY priority").fetchall()))\n    unassigned = conn.execute("SELECT COUNT(*) FROM tickets WHERE assignee_id IS NULL AND status != \'closed\'").fetchone()[0]\n    return {"by_status": by_status, "open_by_priority": by_priority, "unassigned": unassigned}\n')
with open('helpdesk_security.py', 'w', encoding='utf-8') as f:
    f.write('"""Passwords and tokens for the helpdesk (Week 79\'s tools, gathered in one place)."""\nimport base64\nimport hashlib\nimport hmac\nimport secrets\n\n\ndef pbkdf2_sha256(password, salt, iterations, dklen=32):\n    base = hmac.new(password, digestmod=hashlib.sha256)\n    out, block = b"", 1\n    while len(out) < dklen:\n        h = base.copy()\n        h.update(salt + block.to_bytes(4, "big"))\n        u = h.digest()\n        total = int.from_bytes(u, "big")\n        for _ in range(iterations - 1):\n            h = base.copy()\n            h.update(u)\n            u = h.digest()\n            total ^= int.from_bytes(u, "big")\n        out += total.to_bytes(32, "big")\n        block += 1\n    return out[:dklen]\n\n\ndef hash_password(password, iterations):\n    salt = secrets.token_bytes(16)\n    digest = pbkdf2_sha256(password.encode(), salt, iterations)\n    return f"pbkdf2_sha256${iterations}${base64.b64encode(salt).decode()}${base64.b64encode(digest).decode()}"\n\n\ndef check_password(password, stored):\n    try:\n        algorithm, iterations, salt, digest = stored.split("$")\n        expected = pbkdf2_sha256(password.encode(), base64.b64decode(salt), int(iterations))\n        return algorithm == "pbkdf2_sha256" and hmac.compare_digest(expected, base64.b64decode(digest))\n    except ValueError:\n        return False\n\n\ndef issue_token(user_id, expires, key):\n    """A signed bearer token: USER_ID.EXPIRES.SIGNATURE."""\n    payload = f"{user_id}.{expires}"\n    return f"{payload}.{hmac.new(key, payload.encode(), hashlib.sha256).hexdigest()}"\n\n\ndef verify_token(token, key, now):\n    """The user id in a valid, unexpired token, else None."""\n    parts = token.split(".")\n    if len(parts) != 3 or not parts[0].isdigit() or not parts[1].isdigit():\n        return None\n    payload = f"{parts[0]}.{parts[1]}"\n    expected = hmac.new(key, payload.encode(), hashlib.sha256).hexdigest()\n    if not hmac.compare_digest(parts[2], expected) or now >= int(parts[1]):\n        return None\n    return int(parts[0])\n')
with open('helpdesk_app.py', 'w', encoding='utf-8') as f:
    f.write('"""The helpdesk\'s HTTP API (Week 80, Day 3), built with miniflask (Week 77)."""\nimport re\nimport secrets\nimport time\nfrom datetime import datetime, timezone\n\nimport helpdesk_db as db\nfrom helpdesk_domain import ROLES, STATUSES, ValidationError, allowed, can_transition, problem, validate_ticket\nfrom helpdesk_security import check_password, hash_password, issue_token, verify_token\nfrom miniflask import Flask, HTTPException, g, request\n\nERROR_CODES = (400, 401, 403, 404, 405, 409, 415, 422, 500, 503)\n\n\nclass ApiError(HTTPException):\n    def __init__(self, code, detail, **extra):\n        super().__init__(code, detail)\n        self.extra = extra\n\n\ndef iso(t):\n    return datetime.fromtimestamp(t, timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")\n\n\ndef create_app(settings, conn, clock=time.time, log=print):\n    app = Flask(__name__)\n    key = settings["SECRET_KEY"].encode()\n    token_seconds = settings.get("TOKEN_SECONDS", 3600)\n    iterations = settings.get("PASSWORD_ITERATIONS", 600_000)\n    dummy = hash_password(secrets.token_hex(8), iterations)   # checked for unknown emails, so they take as long as real ones\n\n    # ---- step 11\n    for code in ERROR_CODES:\n        @app.errorhandler(code)\n        def handle(e, code=code):\n            if code == 500 and not isinstance(e, HTTPException):\n                incident = secrets.token_hex(4)\n                log(f"incident {incident}: {type(e).__name__}: {e}")\n                return problem(500, "something went wrong", incident=incident)\n            return problem(e.code, e.description, **getattr(e, "extra", {}))\n\n    @app.get("/healthz")\n    def healthz():\n        return {"status": "ok"}\n\n    @app.get("/readyz")\n    def readyz():\n        try:\n            version = conn.execute("SELECT MAX(version) FROM schema_migrations").fetchone()[0]\n        except Exception:\n            version = None\n        if version != len(db.MIGRATIONS):\n            raise ApiError(503, "the database isn\'t ready")\n        return {"status": "ready", "schema": version}\n\n    # ---- step 12\n    @app.before_request\n    def authenticate():\n        g.user = None\n        header = request.headers.get("Authorization", "")\n        if header.startswith("Bearer "):\n            user_id = verify_token(header[7:], key, int(clock()))\n            if user_id is not None:\n                g.user = db.get_user(conn, user_id)\n\n    def user():\n        if g.user is None:\n            raise ApiError(401, "log in first")\n        return g.user\n\n    def body():\n        data = request.get_json(silent=True)\n        if not isinstance(data, dict):\n            raise ApiError(400, "send a JSON object")\n        return data\n\n    @app.post("/users")\n    def signup():\n        data = body()\n        email = data.get("email", "").strip().lower() if isinstance(data.get("email"), str) else ""\n        password = data.get("password")\n        if not re.fullmatch(r"[^@\\s]+@[^@\\s]+\\.[a-z]{2,}", email) or not isinstance(password, str) or not 12 <= len(password) <= 128:\n            raise ApiError(422, "a valid email and a password of 12 to 128 characters are needed")\n        try:\n            user_id = db.create_user(conn, email, hash_password(password, iterations), "customer", iso(clock()))\n        except ValueError:\n            raise ApiError(409, "email already registered") from None\n        return {"id": user_id, "email": email, "role": "customer"}, 201\n\n    @app.post("/tokens")\n    def login():\n        data = body()\n        found = db.get_user_by_email(conn, str(data.get("email", "")).strip().lower())\n        stored = found["password"] if found else dummy\n        if not check_password(str(data.get("password", "")), stored) or found is None:\n            raise ApiError(401, "wrong email or password")\n        expires = int(clock()) + token_seconds\n        return {"token": issue_token(found["id"], expires, key), "expires_at": iso(expires)}, 201\n\n    @app.get("/me")\n    def me():\n        return user()\n\n    # ---- step 13\n    def checked(data, partial=False):\n        try:\n            return validate_ticket(data, partial)\n        except ValidationError as e:\n            raise ApiError(422, "the request has invalid fields", errors=e.errors) from None\n\n    def visible_ticket(ticket_id):\n        me = user()\n        ticket = db.get_ticket(conn, ticket_id)\n        if ticket is None or not allowed(me["role"], "view_ticket", ticket["customer_id"] == me["id"]):\n            raise ApiError(404, f"no ticket {ticket_id}")\n        return me, ticket\n\n    def require(me, action, ticket=None, message="you can\'t do that"):\n        if not allowed(me["role"], action, ticket is not None and ticket["customer_id"] == me["id"]):\n            raise ApiError(403, message)\n\n    @app.post("/tickets")\n    def open_ticket():\n        me = user()\n        require(me, "create_ticket", message="only customers can open tickets")\n        ticket = db.create_ticket(conn, me["id"], checked(body()), iso(clock()))\n        return ticket, 201, {"Location": f"/tickets/{ticket[\'id\']}"}\n\n    @app.get("/tickets")\n    def list_tickets():\n        me = user()\n        args = request.args\n        filters = {}\n        for name, choices in (("status", STATUSES), ("priority", ("low", "normal", "high", "urgent"))):\n            if name in args:\n                if args[name] not in choices:\n                    raise ApiError(400, f"{name} must be one of {\', \'.join(choices)}")\n                filters[name] = args[name]\n        for name, low, high, default in (("after", 0, 10 ** 9, 0), ("limit", 1, 50, 20)):\n            value = args.get(name, str(default))\n            if not value.isdigit() or not low <= int(value) <= high:\n                raise ApiError(400, f"{name} must be a whole number from {low} to {high}")\n            filters[name] = int(value)\n        if not allowed(me["role"], "list_all_tickets"):\n            filters["customer_id"] = me["id"]\n        items, after = db.list_tickets(conn, customer_id=filters.get("customer_id"), status=filters.get("status"),\n                                       priority=filters.get("priority"), q=args.get("q") or None,\n                                       after_id=filters["after"], limit=filters["limit"])\n        return {"items": items, "next": after}\n\n    @app.get("/tickets/<int:ticket_id>")\n    def read_ticket(ticket_id):\n        return visible_ticket(ticket_id)[1]\n\n    # ---- step 14\n    @app.patch("/tickets/<int:ticket_id>")\n    def edit_ticket(ticket_id):\n        me, ticket = visible_ticket(ticket_id)\n        changes = checked(body(), partial=True)\n        if "priority" in changes:\n            require(me, "change_priority", ticket, "only staff can change the priority")\n        if "title" in changes or "body" in changes:\n            require(me, "edit_ticket", ticket, "only the customer who opened it can edit it")\n            if me["role"] == "customer" and ticket["status"] != "open":\n                raise ApiError(409, "only open tickets can be edited")\n        return db.update_ticket(conn, ticket_id, changes, me["id"], iso(clock()))\n\n    @app.post("/tickets/<int:ticket_id>/transitions")\n    def transition(ticket_id):\n        me, ticket = visible_ticket(ticket_id)\n        new = body().get("status")\n        if new not in STATUSES:\n            raise ApiError(422, f"status must be one of {\', \'.join(STATUSES)}")\n        if not can_transition(ticket["status"], new, me["role"]):\n            raise ApiError(403, f"you can\'t move a ticket from {ticket[\'status\']} to {new}")\n        try:\n            return db.change_status(conn, ticket_id, ticket["status"], new, me["id"], iso(clock()))\n        except db.ConflictError:\n            raise ApiError(409, "the ticket changed; reload it and try again") from None\n\n    @app.put("/tickets/<int:ticket_id>/assignee")\n    def assign(ticket_id):\n        me, ticket = visible_ticket(ticket_id)\n        require(me, "assign", ticket, "only staff can assign tickets")\n        agent_id = body().get("agent_id")\n        if not isinstance(agent_id, int) or isinstance(agent_id, bool):\n            raise ApiError(422, "agent_id must be a user id")\n        try:\n            return db.assign(conn, ticket_id, agent_id, me["id"], iso(clock()))\n        except ValueError as e:\n            raise ApiError(422, str(e)) from None\n\n    # ---- step 15\n    @app.delete("/tickets/<int:ticket_id>")\n    def remove_ticket(ticket_id):\n        me, ticket = visible_ticket(ticket_id)\n        require(me, "delete_ticket", ticket, "only admins can delete tickets")\n        db.delete_ticket(conn, ticket_id)\n        return "", 204\n\n    @app.get("/tickets/<int:ticket_id>/comments")\n    def comments(ticket_id):\n        me, ticket = visible_ticket(ticket_id)\n        return {"items": db.list_comments(conn, ticket_id, include_internal=allowed(me["role"], "internal_note"))}\n\n    @app.post("/tickets/<int:ticket_id>/comments")\n    def comment(ticket_id):\n        me, ticket = visible_ticket(ticket_id)\n        require(me, "comment", ticket)\n        data = body()\n        text, internal = data.get("body"), data.get("internal", False)\n        if not isinstance(text, str) or not 1 <= len(text.strip()) <= 5000 or not isinstance(internal, bool):\n            raise ApiError(422, "body must be 1 to 5000 characters and internal true or false")\n        if internal:\n            require(me, "internal_note", ticket, "only staff can add internal notes")\n        return db.add_comment(conn, ticket_id, me["id"], text.strip(), internal, iso(clock())), 201\n\n    @app.get("/stats")\n    def stats():\n        require(user(), "view_stats", message="only staff can see statistics")\n        return db.stats(conn)\n\n    return app\n')

**Predict before you run:** what will this print?



In [ ]:
import json

record = {"request_id": "7b33fd1b", "method": "GET", "path": "/tickets", "status": 200, "ms": 12, "user": 7}
line = json.dumps(record)
print(line)
print(json.loads(line)["status"] == 200, "\n" in line)

### Exercise: Step 21: settings

Write `load_settings(env)` following the table: return a dict with all five settings (the two numbers as `int`s, the log level upper-cased). Collect problems as `"SECRET_KEY is required"`, `"SECRET_KEY must be at least 32 characters"`, `"NAME must be a whole number from LOW to HIGH"` and `"LOG_LEVEL must be DEBUG, INFO, WARNING or ERROR"`, and if there are any, raise `RuntimeError` with them sorted and joined by `"; "`. Then `summary(settings)`: `NAME=VALUE` pairs in name order, joined by `", "`, with the secret key's value shown as `********`.

In [ ]:
def load_settings(env):
    ___


def summary(settings):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_good():\n    \"defaults and conversions\"\n    s = load_settings({\"SECRET_KEY\": \"k\" * 40, \"TOKEN_SECONDS\": \"900\", \"LOG_LEVEL\": \"debug\"})\n    assert s == {\"SECRET_KEY\": \"k\" * 40, \"DATABASE_PATH\": \"helpdesk.db\", \"TOKEN_SECONDS\": 900, \"PASSWORD_ITERATIONS\": 600000, \"LOG_LEVEL\": \"DEBUG\"}\n    assert summary(s) == \"DATABASE_PATH=helpdesk.db, LOG_LEVEL=DEBUG, PASSWORD_ITERATIONS=600000, SECRET_KEY=********, TOKEN_SECONDS=900\"\n    assert load_settings({\"SECRET_KEY\": \"x\" * 32, \"DATABASE_PATH\": \"/data/hd.db\"})[\"DATABASE_PATH\"] == \"/data/hd.db\"\n\ndef test_problems():\n    \"every problem, no values\"\n    try:\n        load_settings({\"SECRET_KEY\": \"short-secret\", \"TOKEN_SECONDS\": \"5\", \"LOG_LEVEL\": \"loud\", \"PASSWORD_ITERATIONS\": \"1e6\"})\n    except RuntimeError as e:\n        assert str(e) == (\"LOG_LEVEL must be DEBUG, INFO, WARNING or ERROR; PASSWORD_ITERATIONS must be a whole number from 100000 to 10000000; \"\n                          \"SECRET_KEY must be at least 32 characters; TOKEN_SECONDS must be a whole number from 60 to 86400\")\n        assert \"short-secret\" not in str(e)\n    else:\n        raise AssertionError(\"must refuse to start\")\n    try:\n        load_settings({})\n    except RuntimeError as e:\n        assert str(e) == \"SECRET_KEY is required\"")

### Exercise: Step 22: request ids and logs

Write `add_request_logging(app, log, timer)`: a `before_request` function that sets `g.request_id` (the incoming `X-Request-ID` if it fully matches `[A-Za-z0-9-]{8,64}`, else `secrets.token_hex(8)`) and `g.started = timer()`; and an `after_request` function that sets the response's `X-Request-ID` header, calls `log(json.dumps({...}))` with the keys `request_id`, `method`, `path` (`request.path`), `status`, `ms` (`round((timer() - g.started) * 1000)`) and `user` (`g.get("user")`'s `id`, or `None`), in that order, and returns the response. Return the app.

In [ ]:
import json
import re
import secrets

from miniflask import g, request


def add_request_logging(app, log, timer):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _app():\n    import helpdesk_db as db\n    from helpdesk_app import create_app\n    conn = db.connect()\n    db.migrate(conn)\n    logs, t = [], [100.0]\n    app = create_app({\"SECRET_KEY\": \"k\" * 40, \"PASSWORD_ITERATIONS\": 50}, conn, lambda: 1800000000, logs.append)\n    return add_request_logging(app, logs.append, lambda: t[0]), logs, t\n\ndef test_ids():\n    \"ids in, ids out\"\n    import json\n    app, logs, t = _app()\n    c = app.test_client()\n    r = c.get(\"/healthz\", headers={\"X-Request-ID\": \"lb-7f3a-91c2\"})\n    assert r.headers[\"X-Request-ID\"] == \"lb-7f3a-91c2\"\n    assert json.loads(logs[-1]) == {\"request_id\": \"lb-7f3a-91c2\", \"method\": \"GET\", \"path\": \"/healthz\", \"status\": 200, \"ms\": 0, \"user\": None}\n    for bad in (\"short\", \"has spaces in it\", \"x\" * 65, \"ok-id-but\\nforged line\"):\n        rid = c.get(\"/healthz\", headers={\"X-Request-ID\": bad}).headers[\"X-Request-ID\"]\n        assert rid != bad and len(rid) == 16\n    assert c.get(\"/healthz\").headers[\"X-Request-ID\"] != c.get(\"/healthz\").headers[\"X-Request-ID\"]\n\ndef test_log_lines():\n    \"one JSON line each, with the user and no query string\"\n    import json\n    app, logs, t = _app()\n    c = app.test_client()\n    c.post(\"/users\", json={\"email\": \"ada@example.org\", \"password\": \"customer password\"})\n    token = c.post(\"/tokens\", json={\"email\": \"ada@example.org\", \"password\": \"customer password\"}).get_json()[\"token\"]\n    r = c.get(f\"/tickets?q=printer&token={token}\", headers={\"Authorization\": \"Bearer \" + token})\n    line = logs[-1]\n    record = json.loads(line)\n    assert record[\"path\"] == \"/tickets\" and record[\"user\"] == 1 and record[\"status\"] == 200 and token not in line\n    assert list(record) == [\"request_id\", \"method\", \"path\", \"status\", \"ms\", \"user\"] and \"\\n\" not in line")

### Exercise: Step 23: the container

Write `lint_dockerfile(text)`: a sorted list of problems (without duplicates). Ignore blank lines and comments; each instruction is the first word (upper-cased) and the rest. Report: `"pin the base image NAME to a version"` for each `FROM` image with no tag or the tag `latest`; `"no secrets in ENV: NAME"` or `"no secrets in ARG: NAME"` when the variable's name (before any `=`) contains `SECRET`, `PASSWORD`, `TOKEN` or `KEY` in any case; `"run as a user other than root"` if there's no `USER` or the last one is `root` or `0`; `"copy requirements and install them before copying the code"` if a `COPY . ...` comes before the first `RUN` containing `pip install`; `"pip install needs --no-cache-dir"` and `"pip install needs --require-hashes"` for a `pip install` without them; `"add a HEALTHCHECK"`; and `"write CMD in exec form, as a JSON list"` if the last `CMD` doesn't start with `[` (or there's none).

In [ ]:
import re

DOCKERFILE = "\n".join([
    'FROM python:3.13-slim',
    'WORKDIR /app',
    'COPY requirements.txt .',
    'RUN pip install --no-cache-dir --require-hashes -r requirements.txt',
    'COPY . .',
    'RUN useradd --create-home helpdesk',
    'USER helpdesk',
    'EXPOSE 8000',
    'HEALTHCHECK CMD python -c "import urllib.request; urllib.request.urlopen(\'http://localhost:8000/healthz\')"',
    'CMD ["gunicorn", "--bind", "0.0.0.0:8000", "helpdesk_wsgi:app"]',
]) + "\n"


def lint_dockerfile(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_good():\n    \"the helpdesk's Dockerfile passes\"\n    assert lint_dockerfile(DOCKERFILE) == []\n\ndef test_bad():\n    \"every problem\"\n    bad = \"\"\"# a quick one\nFROM python\nENV SECRET_KEY=abc PORT=8000\nARG db_password\nCOPY . .\nRUN pip install -r requirements.txt\nCMD gunicorn app:app\n\"\"\"\n    assert lint_dockerfile(bad) == [\"add a HEALTHCHECK\", \"copy requirements and install them before copying the code\", \"no secrets in ARG: db_password\",\n                                    \"no secrets in ENV: SECRET_KEY\", \"pin the base image python to a version\", \"pip install needs --no-cache-dir\",\n                                    \"pip install needs --require-hashes\", \"run as a user other than root\", \"write CMD in exec form, as a JSON list\"]\n    assert lint_dockerfile(DOCKERFILE.replace(\"python:3.13-slim\", \"python:latest\")) == [\"pin the base image python to a version\"]\n    assert lint_dockerfile(DOCKERFILE + \"USER root\\n\") == [\"run as a user other than root\"]\n    assert lint_dockerfile(DOCKERFILE.replace(\"ENV\", \"\").replace(\"EXPOSE 8000\", \"ENV API_TOKEN_URL=x\")) == [\"no secrets in ENV: API_TOKEN_URL\"]")

### Exercise: Step 24: changing a live database

Write `compatibility_problems(sql)`: split a migration's SQL at `;`, and for each non-empty statement (its words upper-cased and single-spaced for matching, but reported as written, stripped) report `"drops something old code uses: STATEMENT"` (it starts with `DROP TABLE`, `DROP COLUMN`, or `ALTER TABLE NAME DROP`), `"renames something old code uses: STATEMENT"` (`ALTER TABLE NAME RENAME`), or `"old code can't insert without the new column: STATEMENT"` (`ALTER TABLE NAME ADD` with `NOT NULL` and no `DEFAULT`). Return the list in statement order. Then `plan_rename(table, old, new, column_type)`: the three releases as `(label, sql_statements, what_the_code_does)` tuples, exactly: `("release 1 (expand)", ["ALTER TABLE T ADD COLUMN NEW TYPE"], "code writes both OLD and NEW, reads OLD")`, `("release 2 (migrate)", ["UPDATE T SET NEW = OLD WHERE NEW IS NULL"], "code writes both OLD and NEW, reads NEW")` and `("release 3 (contract)", ["ALTER TABLE T DROP COLUMN OLD"], "code uses only NEW")`.

In [ ]:
import re


def compatibility_problems(sql):
    ___


def plan_rename(table, old, new, column_type):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_problems():\n    \"safe and breaking statements\"\n    assert compatibility_problems(\"ALTER TABLE tickets ADD COLUMN source TEXT NOT NULL DEFAULT 'web'; CREATE INDEX idx_src ON tickets (source);\") == []\n    assert compatibility_problems(\"ALTER TABLE tickets ADD COLUMN due_at TEXT; CREATE TABLE tags (id INTEGER PRIMARY KEY)\") == []\n    sql = \"\"\"ALTER TABLE tickets RENAME COLUMN body TO description;\n             alter table tickets add column team_id integer not null;\n             DROP TABLE events;\n             ALTER TABLE  comments   DROP COLUMN internal\"\"\"\n    assert compatibility_problems(sql) == [\n        \"renames something old code uses: ALTER TABLE tickets RENAME COLUMN body TO description\",\n        \"old code can't insert without the new column: alter table tickets add column team_id integer not null\",\n        \"drops something old code uses: DROP TABLE events\",\n        \"drops something old code uses: ALTER TABLE  comments   DROP COLUMN internal\"]\n\ndef test_plan():\n    \"expand, migrate, contract, each safe for the code still running\"\n    import sqlite3\n    plan = plan_rename(\"tickets\", \"body\", \"description\", \"TEXT\")\n    assert plan == [(\"release 1 (expand)\", [\"ALTER TABLE tickets ADD COLUMN description TEXT\"], \"code writes both body and description, reads body\"),\n                    (\"release 2 (migrate)\", [\"UPDATE tickets SET description = body WHERE description IS NULL\"], \"code writes both body and description, reads description\"),\n                    (\"release 3 (contract)\", [\"ALTER TABLE tickets DROP COLUMN body\"], \"code uses only description\")]\n    assert compatibility_problems(\";\".join(plan[0][1])) == [] and compatibility_problems(\";\".join(plan[1][1])) == []\n    conn = sqlite3.connect(\":memory:\")\n    conn.execute(\"CREATE TABLE tickets (id INTEGER PRIMARY KEY, body TEXT)\")\n    conn.execute(\"INSERT INTO tickets (body) VALUES ('old row')\")\n    for statement in plan[0][1]:\n        conn.execute(statement)\n    conn.execute(\"INSERT INTO tickets (body) VALUES ('from old code')\")\n    conn.execute(\"INSERT INTO tickets (body, description) VALUES ('new code', 'new code')\")\n    for statement in plan[1][1]:\n        conn.execute(statement)\n    assert [r[0] for r in conn.execute(\"SELECT description FROM tickets ORDER BY id\")] == [\"old row\", \"from old code\", \"new code\"]")

### Exercise: Step 25: the release

Write `next_version(current, changes)`: `changes` is a list of `(kind, description)` with kinds `"breaking"`, `"feature"` or `"fix"`; return the next version text (a breaking change bumps MAJOR, else a feature bumps MINOR, else PATCH; lower numbers reset to 0); an empty list raises `ValueError("nothing to release")`. Then `release_gate(facts)`: a list of reasons the release can't go, in this order: `"CI isn't green"` (unless `facts["ci_green"]` is true); `"N migration statement(s) would break the running version"` (if `migration_problems` is non-empty); `"N serious vulnerability(ies) in dependencies"` (findings in `audit` with severity of at least 7.0); `"no changelog entry for VERSION"` (if `version` isn't in `changelog_versions`); `"the Dockerfile has problems"` (if `dockerfile_problems` is non-empty). Missing keys count as empty or false.

In [ ]:
def next_version(current, changes):
    ___


def release_gate(facts):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_versions():\n    \"semantic versioning\"\n    assert next_version(\"1.4.2\", [(\"fix\", \"a\")]) == \"1.4.3\"\n    assert next_version(\"1.4.2\", [(\"fix\", \"a\"), (\"feature\", \"b\")]) == \"1.5.0\"\n    assert next_version(\"1.4.2\", [(\"feature\", \"b\"), (\"breaking\", \"c\")]) == \"2.0.0\"\n    assert next_version(\"0.9.9\", [(\"feature\", \"x\")]) == \"0.10.0\"\n    try:\n        next_version(\"1.0.0\", [])\n    except ValueError as e:\n        assert str(e) == \"nothing to release\"\n\ndef test_gate():\n    \"every reason\"\n    ok = {\"ci_green\": True, \"version\": \"1.5.0\", \"changelog_versions\": [\"1.4.2\", \"1.5.0\"], \"audit\": [{\"severity\": 5.4}],\n          \"migration_problems\": [], \"dockerfile_problems\": []}\n    assert release_gate(ok) == []\n    bad = {\"ci_green\": False, \"version\": \"1.5.0\", \"changelog_versions\": [\"1.4.2\"], \"audit\": [{\"severity\": 9.8}, {\"severity\": 7.0}, {\"severity\": 6.9}],\n           \"migration_problems\": [\"drops something\", \"renames something\"], \"dockerfile_problems\": [\"add a HEALTHCHECK\"]}\n    assert release_gate(bad) == [\"CI isn't green\", \"2 migration statement(s) would break the running version\",\n                                 \"2 serious vulnerability(ies) in dependencies\", \"no changelog entry for 1.5.0\", \"the Dockerfile has problems\"]\n    assert release_gate({\"version\": \"2.0.0\"}) == [\"CI isn't green\", \"no changelog entry for 2.0.0\"]")

In [ ]:
import json, re, secrets, time
from miniflask import g, request
import helpdesk_db as db
from helpdesk_app import create_app

def load_settings(env):
    """The helpdesk's settings from environment variables; RuntimeError listing every problem."""
    settings, problems = {}, []
    key = env.get("SECRET_KEY")
    if key is None:
        problems.append("SECRET_KEY is required")
    elif len(key) < 32:
        problems.append("SECRET_KEY must be at least 32 characters")
    else:
        settings["SECRET_KEY"] = key
    settings["DATABASE_PATH"] = env.get("DATABASE_PATH", "helpdesk.db")
    for name, default, low, high in (("TOKEN_SECONDS", 3600, 60, 86400), ("PASSWORD_ITERATIONS", 600000, 100000, 10000000)):
        text = env.get(name, str(default))
        if text.isdigit() and low <= int(text) <= high:
            settings[name] = int(text)
        else:
            problems.append(f"{name} must be a whole number from {low} to {high}")
    level = env.get("LOG_LEVEL", "INFO").upper()
    if level in ("DEBUG", "INFO", "WARNING", "ERROR"):
        settings["LOG_LEVEL"] = level
    else:
        problems.append("LOG_LEVEL must be DEBUG, INFO, WARNING or ERROR")
    if problems:
        raise RuntimeError("; ".join(sorted(problems)))
    return settings

def summary(settings):
    """One line for the start-up log, with the secret masked."""
    return ", ".join(f"{k}={'********' if k == 'SECRET_KEY' else v}" for k, v in sorted(settings.items()))

def add_request_logging(app, log, timer):
    """Give every request an id, send it back in X-Request-ID, and log one JSON line per request."""
    @app.before_request
    def start():
        incoming = request.headers.get("X-Request-ID", "")
        g.request_id = incoming if re.fullmatch(r"[A-Za-z0-9-]{8,64}", incoming) else secrets.token_hex(8)
        g.started = timer()

    @app.after_request
    def finish(response):
        response.headers["X-Request-ID"] = g.request_id
        user = g.get("user")
        log(json.dumps({"request_id": g.request_id, "method": request.method, "path": request.path,
                        "status": response.status_code, "ms": round((timer() - g.started) * 1000),
                        "user": user["id"] if user else None}))
        return response
    return app


env = {"SECRET_KEY": "k" * 40, "PASSWORD_ITERATIONS": "100000", "DATABASE_PATH": ":memory:"}
settings = load_settings(env)
print("starting with", summary(settings))
conn = db.connect(settings["DATABASE_PATH"])
print("migrations applied:", db.migrate(conn))
logs = []
app = add_request_logging(create_app({**settings, "PASSWORD_ITERATIONS": 50}, conn, time.time, logs.append), logs.append, time.perf_counter)
client = app.test_client()
for path in ("/readyz", "/tickets", "/nowhere"):
    response = client.get(path)
    record = json.loads(logs[-1])
    print(response.status_code, record["path"], record["status"], len(response.headers["X-Request-ID"]))

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Where we are

The helpdesk is built and tested. In production it runs as several identical **instances** of the app (containers) behind a load balancer, all sharing one database, started by a platform that supplies the environment variables and calls `/readyz` before sending any traffic (Week 77 Day 5). A **release** replaces the instances one at a time, so for a few minutes the **old and new versions run side by side** against the same database. Hold on to that fact: it's what Step 24 is about.

A log line in JSON can be searched and counted by machines ("every 500 in the last hour", "everything request `7b33fd1b` did"), and `json.dumps` escapes newlines, so a hostile value can't forge an extra line (Week 79).

## Step 21: settings

Week 79's rule: configuration comes from the environment, is checked **once, at start-up**, and the service refuses to start with a list of **every** problem, rather than failing later at 3 a.m. on the first request that needs the missing value. The helpdesk's settings:

| Variable | Rule | Default |
|---|---|---|
| `SECRET_KEY` | required, at least 32 characters | |
| `DATABASE_PATH` | any path | `helpdesk.db` |
| `TOKEN_SECONDS` | whole number from 60 to 86400 | `3600` |
| `PASSWORD_ITERATIONS` | whole number from 100000 to 10000000 | `600000` |
| `LOG_LEVEL` | `DEBUG`, `INFO`, `WARNING` or `ERROR` (any case) | `INFO` |

The start-up log gets **one line** listing the settings in use, with the secret masked, so when something behaves oddly the first question ("which settings is it running with?") has an answer in the log.

## Step 22: request ids and logs

When a customer reports "it said something went wrong at about ten past nine", you need to find **that** request among thousands. So every request gets a **request id**:

- if the request arrives with an `X-Request-ID` header (load balancers and other services often add one), use it, but only if it looks like an id (letters, digits and hyphens, 8 to 64 characters), because it will be written into logs and headers (Week 79);
- otherwise make one with `secrets.token_hex(8)`;
- send it back in the response's `X-Request-ID` header, so a client (or the support person) can quote it;
- write **one JSON log line** per request with the id, method, path, status, duration in milliseconds and user id. The **path only**, never the query string or headers, which can hold tokens and personal data.

The duration uses an injected **timer** (in production `time.perf_counter`, which measures elapsed time and never jumps the way the wall clock can), so tests can control it. Week 77's `500` handler already logs an incident id; with request ids in the same log, the two can be matched.

## Step 23: the container

The helpdesk ships as a **container image** (Week 77): a Dockerfile describes how to build it, and every instance runs the same image. A good Dockerfile for a Python service:

```dockerfile
FROM python:3.13-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir --require-hashes -r requirements.txt
COPY . .
RUN useradd --create-home helpdesk
USER helpdesk
EXPOSE 8000
HEALTHCHECK CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/healthz')"
CMD ["gunicorn", "--bind", "0.0.0.0:8000", "helpdesk_wsgi:app"]

```

The habits it follows, each one a common review comment:

- **Pin the base image** (`python:3.13-slim`, not `python` or `python:latest`), so a rebuild next month gets the same Python. (Pinning to an image digest is stricter still.)
- **No secrets** in `ENV` or `ARG`: they're stored in the image's layers, readable by anyone who can pull it. Secrets arrive at run time from the platform (Week 79).
- **Copy the requirements and install them before copying the code**, so Docker's layer cache reuses the slow install step whenever only the code changed.
- `pip install` with `--require-hashes` (Week 79) and `--no-cache-dir` (no point keeping pip's cache in the image).
- **Don't run as root**: create a user and switch to it with `USER`, so a compromised app can't change the container's own files.
- A **`HEALTHCHECK`**, so the container runtime knows when the process is stuck.
- **`CMD` in exec form** (a JSON list), so the server is the container's main process and receives the stop signal directly, shutting down cleanly instead of being killed.

## Step 24: changing a live database

During a release, old and new instances run **at the same time** against one database, and the migration runs **before** any new instance starts. So every migration must leave a database that the **old** code can still use. That rules out, in a single release:

- **dropping** a table or column the old code reads;
- **renaming** a table or column (to the old code, that's a drop);
- **adding a `NOT NULL` column without a default** (the old code's `INSERT`s don't mention it, so they all fail).

Adding tables, adding nullable columns or columns with defaults, and creating indexes are all safe.

So how do you rename a column? In steps, across releases, a pattern called **expand and contract**:

1. **Expand**: add the new column; release code that **writes both** columns and still reads the old one.
2. **Migrate**: copy the old values into the new column for existing rows; release code that **reads the new** column (still writing both, in case of a rollback).
3. **Contract**: once no running code reads the old column, release code that uses only the new one, then drop the old column.

Each step is safe on its own, and you can stop or roll back between any two. It takes three releases instead of one; that's the price of never being down.

## Step 25: the release

Two last decisions before a release goes out: what it's **called**, and whether it **may go**.

**Semantic versioning** (Week 73), for an API: `MAJOR.MINOR.PATCH`. A **breaking** change (removing an endpoint or field, changing what a status means) bumps MAJOR and resets the others; a new **feature** that existing clients can ignore bumps MINOR; only **fixes** bump PATCH. The changelog's entries are labelled with their kind, so the next version can be worked out rather than argued about.

A **release gate** is the list of things that must be true before anything ships, checked by code so nobody skips one under pressure: CI is green; no migration would break the running version (Step 24); no serious vulnerability (CVSS 7.0 or more, Week 79) in the dependencies; the changelog has an entry for this version (Week 73); and the Dockerfile is clean (Step 23). The gate returns **every** reason at once, like every other check this week.

## Checkpoint

Start-up, as the platform would run it: settings checked, migrations applied, the summary logged, and the first requests traced.

## Quiz

## Before you go

On your own computer, build the helpdesk's image (`docker build -t helpdesk .`) and run it with a secret from your environment (`docker run -e SECRET_KEY -p 8000:8000 helpdesk`), then call `/readyz`. Tomorrow is the **project review**: the whole system in one place, its weak points, and one more feature added the safe way.